In [1]:
import os
import glob
import shutil


In [2]:
# 1. route dir
fasta_dir = os.path.join("ProAffinity-GNN", "data", "FASTA", "mixed")
mole2_dir = os.path.join("ProAffinity-GNN", "data", "FASTA", "2mole")
mole3_dir = os.path.join("ProAffinity-GNN", "data", "FASTA", "3mole")
chain_index_path = os.path.join("ProAffinity-GNN", "data", "chain_index.txt")

os.makedirs(mole2_dir, exist_ok=True)
os.makedirs(mole3_dir, exist_ok=True)

fasta_files = glob.glob(os.path.join(fasta_dir, "*.fasta"))

# delete orginal files
clean_fasta_files = [f for f in fasta_files if os.path.getmtime(f) > 1723000000] # 只保留今天產生的

pdb_chains = {}
file_group = {}

In [3]:
print("Debugging: wrong FASTA begginings, etc...")

# Task A: wrong FASTA begginings
for file in clean_fasta_files:
    filename = os.path.basename(file)
    pdb_id = filename.split('_')[0].lower() 
    
    with open(file, 'r') as f:
        lines = f.readlines()
        
    header = lines[0].strip()
    
    # from >1A22:A to chain_id (A)
    if ":" in header:
        chain_id = header.split(":")[1]
        # Author's format:  >1A22_1|Chain A|Fake Protein|Fake Species
        # split('|')[1] will target Chain A
        lines[0] = f">{filename.replace('.fasta', '')}|Chain {chain_id}|Fake Protein|Fake Species\n"
        
        with open(file, 'w') as f:
            f.writelines(lines)
    else:
        # author's file
        chain_str = header.split("|")[1]
        chain_id = chain_str[-2] if chain_str[-1] == ']' else chain_str[-1]

    # get each pdb and seq
    if pdb_id not in file_group:
        file_group[pdb_id] = []
        pdb_chains[pdb_id] = []
    file_group[pdb_id].append(file)
    pdb_chains[pdb_id].append(chain_id)

# Task B: Move the files and generate corresponding list
with open(chain_index_path, 'w') as f:
    for pdb, files in file_group.items():
        # 2mole or 3mole
        target_dir = mole2_dir if len(files) == 2 else mole3_dir
        
        for file in files:
            shutil.copy(file, os.path.join(target_dir, os.path.basename(file)))
            
        # write in chain_index.txt, format for two strings: "A; B;"
        chains = pdb_chains[pdb]
        if len(chains) == 2:
            chain_str = f"{chains[0]}; {chains[1]};"
        else:
            chain_str = "; ".join(chains) + ";" 
            
        f.write(f"{pdb}\t{chain_str}\n")

print(f"\n Fixed {len(file_group)} pdbs")
print(f" files moved to 2 moles and 3 moles")
print(f"Sequence list saved in: {chain_index_path}")

Debugging: wrong FASTA begginings, etc...

 Fixed 1270 pdbs
 files moved to 2 moles and 3 moles
Sequence list saved in: ProAffinity-GNN\data\chain_index.txt
